# Brand Studio

A brand kit lives in Pixeltable tables: each brand's palette, type spec, font files, logos, and style guide. The creative ideas are rows too, and so is each campaign. Insert a campaign, and Pixeltable designs one set for it, photographs each idea, puts the brand names in two ways (with an image model and with code), and reviews every image against the brand rules. Every prompt is a column next to the image it made.

## Start from an empty `studio` directory

Start Jupyter with `uv run jupyter lab` in this folder. The models are Gemini, so you need `GEMINI_API_KEY`. Images and video cost money, so each cell that makes them says how many.

In [ ]:
!uv sync

In [ ]:
import pixeltable as pxt
import pixeltable.functions as pxtf

import sample_data as data

# Start over: uncomment to drop every table from an earlier run.
# pxt.drop_dir('studio', force=True, if_not_exists='ignore')

## ORIEL's brand kit, in Pixeltable

ORIEL makes complexion and skin care. Its whole kit is one row in `brands`: palette, type spec, the font files, logos, and the style guide. Pixeltable drew the brand board from the kit, and Gemini read the style guide once into art direction.

In [ ]:
brands = pxt.get_table('studio.brands')
products = pxt.get_table('studio.products')
library = pxt.get_table('studio.library')
brands.where(brands.brand == 'oriel').select(brands.name, brands.category, brands.logo_dark, brands.brand_board).collect()

In [ ]:
brands.where(brands.brand == 'oriel').select(brands.palette, brands.art_direction).collect()

Its products each have one packshot, so every shot shows the same product, and its library holds the brand's own photography.

In [ ]:
products.where(products.brand == 'oriel').select(products.name, products.packshot).collect()

In [ ]:
library.where(library.brand == 'oriel').select(library.note, library.image).collect()

## brume's brand kit

brume makes hair care, and its kit looks nothing like ORIEL's: cool greens and greys, and a lowercase typewriter wordmark. Same tables, different rows.

In [ ]:
brands.where(brands.brand == 'brume').select(brands.name, brands.category, brands.logo_dark, brands.brand_board).collect()

In [ ]:
products.where(products.brand == 'brume').select(products.name, products.packshot).collect()

## The partnership: ORIEL × brume, The Morning Edit

The campaign is one row: ORIEL leads, brume partners, and the vision is the brief plus a look and the ideas to shoot.

In [ ]:
morning_edit = next(c for c in data.CAMPAIGNS if c['campaign'] == 'morning-edit')
{k: morning_edit[k] for k in ('brand', 'partner', 'brief', 'direction', 'moments')}

## Build it yourself

Everything above is rows and columns. The rest of the notebook builds it from an empty `studio` directory: apply the schema, insert the brand kits, and insert a campaign.

## Apply the schema

`pxt schema diff` shows the plan, and `pxt schema update` creates the tables.

In [ ]:
!pxt schema diff schema.py studio

In [ ]:
!pxt schema update schema.py studio -f

Shared wording is rows too: every sentence a model reads that is not part of a direction or a moment.

In [ ]:
prompts = pxt.get_table('studio.prompts')
prompts.insert(data.PROMPTS)
prompts.select(prompts.name, prompts.template).collect()

## The brand kit is rows

Each brand row holds the kit itself, including the font files. On insert, Gemini reads the style guide into art direction (one text call per brand), and Pillow draws a brand board from the palette, fonts, and logo.

In [ ]:
brands = pxt.get_table('studio.brands')
brands.insert([data.brand_row(slug) for slug in ('oriel', 'brume')])
brands.select(brands.name, brands.brand_board).collect()

In [ ]:
brands.select(brands.name, brands.art_direction_prompt, brands.art_direction).collect()

## Products and photography

Each product gets one packshot, so every shot shows the same product: 4 images. The library photos get an embedding index, so a campaign can find the photos closest to its brief.

In [ ]:
library = pxt.get_table('studio.library')
products = pxt.get_table('studio.products')
library.insert(data.library_rows())
products.insert([p for p in data.PRODUCTS if p['brand'] in ('oriel', 'brume')])
products.select(products.name, products.packshot_prompt, products.packshot).collect()

## The creative library

A direction is a look: its prompt templates and house rules. A moment is an idea for putting the brand names into the world, with how the brand names are physically made in it. No model calls here.

In [ ]:
directions = pxt.get_table('studio.directions')
moments = pxt.get_table('studio.moments')
directions.insert([d for d in data.DIRECTIONS if d['direction'] != 'english-countryside'])
moments.insert([m for m in data.MOMENTS if m['moment'] not in data.OUTDOOR_MOMENTS])
moments.select(moments.moment, moments.subject, moments.physics, moments.composite).collect()

## A collaboration is one row

ORIEL × brume: the row names the lead brand, the partner, a direction, products, and four moments. Gemini designs one set for the whole campaign, then photographs it empty. Each moment makes two images, a blank base and the AI letters, and two reviews: 9 images in all. Clips stay off for now.

In [ ]:
campaigns = pxt.get_table('studio.campaigns')
morning_edit = next(c for c in data.CAMPAIGNS if c['campaign'] == 'morning-edit')
campaigns.insert([dict(morning_edit, clips=False)])
campaigns.select(campaigns.world_prompt, campaigns.world.set, campaigns.plate).collect()

## Each shot, two ways

`shots` has one row per campaign and moment. `base` is the shot with its surface blank. `still` is Gemini's edit of that base with the brand names added. `composite` is the same base with the exact logo files set in by code, for the mosaic floors. `version` is the one that passed review.

In [ ]:
shots = pxt.get_table('studio.shots')
shots.select(shots.moment, shots.base, shots.still, shots.composite, shots.version).collect()

Every prompt is stored, so you can read exactly what each image was asked for, and why each review passed or failed.

In [ ]:
shots.select(shots.moment, shots.base_prompt, shots.letter_prompt).collect()

In [ ]:
shots.select(shots.moment, ai=shots.review.blocking, code=shots.composite_review.blocking).collect()

## Add a brand: wallow

A new brand is one more brands row, from its own kit: palette, type spec, fonts, logo, and style guide. Its products are rows too: 2 packshots.

In [ ]:
brands.insert([data.brand_row('wallow')])
products.insert([p for p in data.PRODUCTS if p['brand'] == 'wallow'])
products.where(products.brand == 'wallow').select(products.name, products.packshot).collect()

## ORIEL × wallow, with a new vision

A new look is one more directions row: here, a clean vintage clawfoot tub outdoors in the English countryside, with fresh flowers and gingham. Its outdoor moments are rows too. Then the campaign row picks ORIEL, wallow, the new direction, and the moments: 1 set photo and 4 shots, 9 images.

In [ ]:
directions.insert([d for d in data.DIRECTIONS if d['direction'] == 'english-countryside'])
moments.insert([m for m in data.MOMENTS if m['moment'] in data.OUTDOOR_MOMENTS])
countryside = next(c for c in data.CAMPAIGNS if c['campaign'] == 'oriel-wallow-countryside')
campaigns.insert([countryside])
shots.where(shots.campaign == 'oriel-wallow-countryside').select(shots.moment, shots.base, shots.still, shots.version).collect()

## Change an idea, then recompute only its shots

Each shot copied its moment when it was made. After you change a moment, recompute `spec` for the shots that use it. Everything built on `spec` reruns for those rows, and every other row keeps its results: here, 1 shot and 2 images.

In [ ]:
moments.update(
    {'treatment': 'Mosaic lettering across the floor, spelled out in small hexagon tiles in {letter_colors}, very large: {wordmarks}.'},
    where=moments.moment == 'hex_floor',
)
shots.recompute_columns('spec', where=shots.moment == 'hex_floor')
shots.where(shots.moment == 'hex_floor').select(shots.campaign, shots.still, shots.composite, shots.version).collect()

## Turn on clips

Set `clips` on the collab. Gemini Omni animates only the shots that passed review, and Gemini watches each clip for glitches.

In [ ]:
campaigns.update({'clips': True}, where=campaigns.campaign == 'morning-edit')
shots.where(shots.campaign == 'morning-edit').select(shots.moment, shots.clip, shots.clip_approved).collect()

## Cut the reel

A reel is one more row: the campaign and the moments in cut order. It uses only the clips that passed review.

In [ ]:
reels = pxt.get_table('studio.reels')
reels.insert(data.REELS)
reels.select(reels.reel, reels.video).collect()